# Práctica: fine-tuning de un modelo para entender órdenes de un asistente virtual

En el notebook **48 (Fine-tuning de DistilBERT)** ajustamos un modelo preentrenado en español para decidir si una reseña es **positiva o negativa**. En esta práctica aplicarás el mismo procedimiento a otro problema muy común en la industria: los asistentes virtuales (Alexa, Siri, los chatbots de los bancos) tienen que saber **qué quiere hacer el usuario** a partir de lo que escribe o dice. A eso se le llama **detección de intención**.

| El usuario dice... | Intención |
|---|---|
| "despiértame a las seis y media" | `poner_alarma` |
| "¿va a llover mañana en Arequipa?" | `consultar_clima` |
| "pídeme un taxi al aeropuerto" | `pedir_taxi` |

Usaremos **MASSIVE**, un dataset público de Amazon con frases reales dirigidas a un asistente de voz, en español. Nos quedaremos con **8 intenciones**.

Diferencias con el notebook 48:

- Ahora hay **8 clases** en lugar de 2, así que cambian la capa de salida del modelo y la forma de calcular el F1.
- Vamos a comparar el fine-tuning con una **línea base clásica** (TF-IDF + regresión logística) para saber si de verdad vale la pena.
- Al final veremos qué hace el modelo cuando le hablan de algo que **no está entre sus 8 intenciones**.

| Ejercicio | Qué harás |
|---|---|
| 1 | Preparar los datos: filtrar las intenciones y codificar las etiquetas |
| 2 | Elegir el modelo base mirando el tokenizador |
| 3 | Entrenar una línea base con TF-IDF |
| 4 | Adaptar las funciones de entrenamiento a varias clases |
| 5 | Fine-tuning con pocos datos y con todos los datos |
| 6 | Análisis de errores con la matriz de confusión |
| 7 | Guardar el modelo, usarlo y manejar frases fuera de tema |

### Cómo trabajar esta práctica

- Reemplaza los `None` y los `...` por tu código. Las celdas que dicen **"ya resuelta"** solo hay que ejecutarlas.
- Cada ejercicio trae **pistas**. Si te trabas, vuelve al notebook 48: casi todo el código tiene un equivalente allí.
- Responde las preguntas de **"Para analizar"** en las celdas **"Tu respuesta"**.
- Entrenar todo toma unos pocos minutos en una laptop con MPS o GPU. En CPU puede tardar alrededor de 10 minutos.

## Preparación

Celda ya resuelta: librerías, semilla y dispositivo, igual que en el notebook 48.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, pipeline, set_seed)

SEED = 42
set_seed(SEED)
pd.set_option("display.max_colwidth", 100)

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"DEVICE: {DEVICE}")

MODELO_INGLES = "distilbert-base-uncased"
MODELO_ESPANOL = "dccuchile/distilbert-base-spanish-uncased"

/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE: mps


## Los datos: MASSIVE en español

Cada fila tiene:

- `text`: lo que el usuario le dijo al asistente.
- `label`: la intención, en inglés (`alarm_set`, `play_music`...). Hay 60 intenciones distintas.
- `split`: si la frase es de entrenamiento (`train`) o de prueba (`test`). El dataset ya viene separado, así que no hace falta `train_test_split`.

La celda descarga los datos de Hugging Face la primera vez (menos de 1 MB) y los guarda en `data/intenciones_asistente_es.csv`. Celda ya resuelta.

In [3]:
DATA_PATH = Path("../data/intenciones_asistente_es.csv")

if not DATA_PATH.exists():
    print("Descargando el dataset...")
    url = "https://huggingface.co/datasets/mteb/MassiveIntentClassification/resolve/main/es/{}-00000-of-00001.parquet"
    partes = [pd.read_parquet(url.format(parte)).assign(split=parte) for parte in ["train", "test"]]
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.concat(partes, ignore_index=True).to_csv(DATA_PATH, index=False)

df = pd.read_csv(DATA_PATH)
print(f"Frases: {len(df)} | intenciones: {df['label'].nunique()}")
df.sample(8, random_state=SEED)

Frases: 14488 | intenciones: 60


,text,label,split
8710,deberías saber sobre,qa_factoid,train
11899,apaga las luces ahora,iot_hue_lightoff,test
2366,cómo está el tiempo hoy en burgos,weather_query,train
6292,me gustaría escuchar noventa y nueve punto cinco,play_radio,train
169,solo muéstrame alguna noticias de b. b. c.,news_query,train
2034,qué hay de nuevo en el mercado de valores,qa_stock,train
10078,cuál es el tipo de cambio de la moneda inglesa al yen,qa_currency,train
1944,hora de madrid,datetime_query,train


## Ejercicio 1. Prepara los datos

Nos quedamos con las 8 intenciones del diccionario `INTENCIONES`, que traduce el nombre original al español. Algunas se parecen a propósito (poner una alarma y agendar un evento; buscar una receta y pedir comida) para que el problema no sea demasiado fácil.

a) Crea `df_int`: solo las filas cuyo `label` esté en `INTENCIONES`, con una columna nueva `intencion` que tenga el nombre en español.

b) Crea `ETIQUETAS`, un diccionario `{0: "agendar_evento", 1: "apagar_luces", ...}` con las intenciones **en orden alfabético**, y `ETIQUETA_A_ID`, el diccionario inverso (`{"agendar_evento": 0, ...}`). Luego **reemplaza** la columna `label` de `df_int` por el número de cada intención. El `Trainer` necesita que las etiquetas sean números de 0 a 7.

c) Separa `df_int` en `train_df` y `test_df` usando la columna `split`. Quédate solo con las columnas `text`, `intencion` y `label`, y usa `reset_index(drop=True)`.

**Pistas:**
- a) `df["label"].isin(INTENCIONES)` devuelve `True` en las filas cuya etiqueta es una clave del diccionario. Después, `.map(INTENCIONES)` traduce cada etiqueta.
- b) `sorted(INTENCIONES.values())` da los nombres en orden y `enumerate(...)` les pone un número. Para invertir un diccionario: `{valor: clave for clave, valor in d.items()}`.
- c) Filtra con `df_int["split"] == "train"`.

In [ ]:
INTENCIONES = {
    "alarm_set": "poner_alarma",
    "calendar_set": "agendar_evento",
    "weather_query": "consultar_clima",
    "play_music": "poner_musica",
    "cooking_recipe": "buscar_receta",
    "takeaway_order": "pedir_comida",
    "transport_taxi": "pedir_taxi",
    "iot_hue_lightoff": "apagar_luces",
}

# a)
df_int = None
df_int["intencion"] = None

# b)
ETIQUETAS = None
ETIQUETA_A_ID = None
df_int["label"] = None

# c)
columnas = ["text", "intencion", "label"]
train_df = None
test_df = None

print(ETIQUETAS)
print(f"\nEntrenamiento: {len(train_df)} | prueba: {len(test_df)}")
print("\nFrases de entrenamiento por intención:")
print(train_df["intencion"].value_counts())

Celda ya resuelta: un ejemplo de cada intención.

In [ ]:
train_df.groupby("intencion").sample(1, random_state=SEED)

**Para analizar**

1. ¿Las clases están balanceadas? ¿Qué accuracy tendría un modelo que respondiera siempre la intención más frecuente?
2. En el notebook 48 usamos `f1_score` con 2 clases. Con 8 clases desbalanceadas, ¿por qué conviene mirar el **F1 macro** (el promedio del F1 de cada clase) además del accuracy?

**Tu respuesta:**

*(Escribe aquí tu respuesta)*

## Ejercicio 2. Elige el modelo base

En el notebook 48 vimos que el fine-tuning solo funciona si el modelo base conoce el idioma. Compruébalo con estas órdenes:

Para cada modelo (`MODELO_INGLES` y `MODELO_ESPANOL`) carga su tokenizador y muestra cuántos tokens produce cada frase y cuáles son.

**Pista:** `AutoTokenizer.from_pretrained(nombre).tokenize(frase)` devuelve la lista de tokens.

In [ ]:
ordenes = [
    "despiértame mañana a las seis",
    "apaga las luces de la habitación",
    "pídeme un ceviche a domicilio",
]

for nombre in [MODELO_INGLES, MODELO_ESPANOL]:
    tokenizer = ...
    print(nombre)
    for frase in ordenes:
        tokens = ...
        print(f"  {len(tokens):>2} tokens  {tokens}")
    print()

**Para analizar**

1. ¿Qué modelo parte las palabras en más trozos? ¿Qué palabras reconoce completas el modelo en español?
2. ¿Con cuál de los dos harás el fine-tuning y por qué?

**Tu respuesta:**

*(Escribe aquí tu respuesta)*

## Ejercicio 3. Línea base: TF-IDF + regresión logística

Antes de entrenar una red de 66 millones de parámetros conviene saber cuánto se consigue con algo simple. Si la línea base ya acierta casi todo, el fine-tuning quizá no valga la pena.

a) Completa `entrenar_tfidf(datos)`: crea un pipeline con `TfidfVectorizer(ngram_range=(1, 2))` y `LogisticRegression(max_iter=1000)`, entrénalo con las columnas `text` y `label` de `datos` y devuélvelo.

b) Completa `metricas(etiquetas, predicciones)`: devuelve un diccionario con el `accuracy` y el `f1_macro`, redondeados a 3 decimales. La usaremos con todos los modelos.

c) Entrena la línea base con `train_df` y evalúala en `test_df`.

**Pistas:**
- `make_pipeline(paso1, paso2)` encadena el vectorizador y el clasificador; después basta con `.fit(textos, etiquetas)` y `.predict(textos)`.
- Para el F1 de varias clases: `f1_score(etiquetas, predicciones, average="macro")`.

In [ ]:
# a)
def entrenar_tfidf(datos):
    modelo = ...
    ...
    return modelo


# b)
def metricas(etiquetas, predicciones):
    return {
        "accuracy": ...,
        "f1_macro": ...,
    }


# c)
linea_base = None
metricas(test_df["label"], linea_base.predict(test_df["text"]))

## Ejercicio 4. Funciones de fine-tuning para varias clases

Las funciones del notebook 48 estaban pensadas para 2 clases. Adáptalas:

a) En `calcular_metricas`, convierte los logits en predicciones (la clase con el logit más alto) y usa la función `metricas` del ejercicio 3.

b) En `entrenar`, carga el modelo con una capa de clasificación de **tantas salidas como intenciones**, y pásale `id2label` y `label2id` para que responda con el nombre de la intención y no con `LABEL_3`.

`tokenizar` y `predecir_etiquetas` ya están resueltas. El resto de `entrenar` es igual al del notebook 48.

**Pistas:**
- a) `np.argmax(logits, axis=-1)`.
- b) `num_labels=len(ETIQUETAS)`, `id2label=ETIQUETAS`, `label2id=ETIQUETA_A_ID`.

In [ ]:
def tokenizar(dataset, tokenizer):
    return dataset.map(lambda lote: tokenizer(lote["text"], truncation=True, max_length=64), batched=True)


# a)
def calcular_metricas(eval_pred):
    logits, etiquetas = eval_pred
    predicciones = ...
    return metricas(etiquetas, predicciones)


# b)
def entrenar(datos, epocas, batch_size=16, learning_rate=5e-5, nombre_modelo=MODELO_ESPANOL):
    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
    model = AutoModelForSequenceClassification.from_pretrained(
        nombre_modelo,
        num_labels=...,
        id2label=...,
        label2id=...,
    )

    training_args = TrainingArguments(
        output_dir=".results",
        num_train_epochs=epocas,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        warmup_steps=0.1,
        weight_decay=0.01,
        logging_strategy="epoch",
        save_strategy="no",
        report_to="none",
        seed=SEED,
    )

    dataset = Dataset.from_pandas(datos[["text", "label"]], preserve_index=False)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenizar(dataset, tokenizer),
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=calcular_metricas,
    )
    trainer.train()
    return trainer, tokenizer


def predecir_etiquetas(trainer, tokenizer, datos):
    dataset = Dataset.from_pandas(datos[["text", "label"]], preserve_index=False)
    logits = trainer.predict(tokenizar(dataset, tokenizer)).predictions
    return np.argmax(logits, axis=-1)

## Ejercicio 5. ¿Cuándo vale la pena el fine-tuning?

En la vida real casi nunca tenemos miles de frases etiquetadas al empezar un proyecto. Vamos a simular dos situaciones:

- **Pocos datos**: 16 frases por intención (128 en total), lo que alguien podría escribir a mano en una tarde.
- **Todos los datos**: el `train_df` completo.

a) Crea `pocos_df` con 16 frases de cada intención tomadas de `train_df`.

b) Para cada conjunto de datos, entrena la línea base TF-IDF y el modelo con fine-tuning, y guarda en `resultados` una fila por modelo con sus métricas en `test_df`. Usa **10 épocas y `batch_size=8`** con pocos datos y **3 épocas** con todos los datos (como en el notebook 48: con más datos hacen falta menos pasadas).

c) Muestra `resultados` como DataFrame.

**Pistas:**
- a) `train_df.groupby("label").sample(n=16, random_state=SEED)`.
- b) Para el modelo con fine-tuning, `predecir_etiquetas(trainer, tokenizer, test_df)` devuelve las predicciones que le pasas a `metricas`.
- b) `{"modelo": ..., "datos": ..., **metricas(...)}` junta todo en un diccionario.

In [ ]:
# a)
pocos_df = None

# b)
resultados = []
modelos = {}

for nombre_datos, datos, epocas, batch_size in [("16 por intención", pocos_df, 10, 8),
                                                 ("train completo", train_df, 3, 16)]:
    tfidf = ...
    resultados.append(...)

    trainer, tokenizer = ...
    modelos[nombre_datos] = (trainer, tokenizer)
    resultados.append(...)

# c)
pd.DataFrame(resultados)

**Para analizar**

1. Con 16 frases por intención, ¿cuántos puntos de F1 macro le saca el fine-tuning a la línea base? ¿Y con todos los datos?
2. ¿Por qué crees que la ventaja del modelo preentrenado es mayor cuando hay pocos datos? Piensa en qué sabe cada modelo antes de ver nuestras frases.
3. Si tuvieras que lanzar el asistente mañana y solo tuvieras 128 frases, ¿qué modelo usarías?

**Tu respuesta:**

*(Escribe aquí tu respuesta)*

## Ejercicio 6. Análisis de errores

Un número como el accuracy no dice **en qué** se equivoca el modelo. Usa el modelo entrenado con el train completo:

a) Calcula `predicciones` para `test_df`.

b) Dibuja la matriz de confusión con los nombres de las intenciones.

c) Crea `errores`: las frases de `test_df` mal clasificadas, con una columna `prediccion` con el nombre de la intención predicha. Muestra las primeras 15.

**Pistas:**
- b) `ConfusionMatrixDisplay.from_predictions(reales, predichas, display_labels=list(ETIQUETAS.values()), xticks_rotation=45, ax=ax)`.
- c) `test_df.assign(prediccion=...)` agrega la columna; filtra las filas donde `label` no coincide con `predicciones`. Para pasar de número a nombre: `pd.Series(predicciones).map(ETIQUETAS)`.

In [ ]:
trainer_final, tokenizer_final = modelos["train completo"]

# a)
predicciones = None

# b)
fig, ax = plt.subplots(figsize=(8, 7))
...
ax.set_title("Matriz de confusión en test")
plt.tight_layout()
plt.show()

# c)
errores = None
print(f"Errores: {len(errores)} de {len(test_df)}")
errores[["text", "intencion", "prediccion"]].head(15)

**Para analizar**

1. ¿Qué pares de intenciones se confunden más? ¿Tiene sentido por lo que dicen las frases?
2. Lee los errores. ¿En cuántos dirías que el modelo se equivocó de verdad y en cuántos la frase es ambigua o la etiqueta original es discutible?

**Tu respuesta:**

*(Escribe aquí tu respuesta)*

## Ejercicio 7. Guarda el modelo y úsalo

a) Guarda el modelo final **y su tokenizador** en `../data/modelo_intenciones` y cárgalo con `pipeline("text-classification", ...)`.

b) Prueba el clasificador con las frases de `mis_frases` (agrega 2 o 3 tuyas). Las últimas no corresponden a ninguna de las 8 intenciones.

c) Un asistente real no debe ejecutar una acción si no está seguro. Completa `responder(texto, umbral)`: si la probabilidad de la intención predicha es menor que `umbral`, debe devolver `"no_entendi"`; si no, la intención.

**Pistas:**
- a) Igual que en el notebook 48: `trainer.save_model(ruta)` y `tokenizer.save_pretrained(ruta)`.
- b) `clasificador(texto)[0]` devuelve un diccionario con `label` y `score`.

In [ ]:
# a)
RUTA_MODELO = "../data/modelo_intenciones"
...

clasificador = None

# b)
mis_frases = [
    "levántame a las 5 que tengo vuelo",
    "pon algo de salsa para cocinar",
    "cómo preparo un lomo saltado",
    "tráeme un pollo a la brasa con papas",
    "necesito un taxi a miraflores",
    "anota la reunión con el contador el viernes",
    "¿hará frío esta noche en cusco?",
    "cuánto es 25 por 4",
    "cuéntame un chiste",
    "¿quién ganó el partido de ayer?",
]

for texto in mis_frases:
    resultado = ...
    print(f"{resultado['label']:>16} ({resultado['score']:.2f})  {texto}")

In [ ]:
# c)
def responder(texto, umbral=0.7):
    resultado = ...
    ...


for texto in mis_frases:
    print(f"{responder(texto):>16}  {texto}")